In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

import string
from collections import Counter
import time

In [2]:
def get_device():
    if torch.backends.mps.is_available():
        print("MPS is available")
        return torch.device("mps")
    elif torch.backends.cuda.is_available():
        print("CUDA is available")
        return torch.device("cuda")
    else:
        print("CPU is available")
        return torch.device("cpu")

In [ ]:
EMBEDDING_SIZE = 256
WINDOW_SIZE = 2
BATCH_SIZE = 128

MODEL_FILE = "cbow_model_test.pt"

DEVICE = get_device()  # torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print('Using device:', str(DEVICE).upper(), "\n")

In [ ]:
def custom_standardization(text: str) -> str:
    text = text.lower()
    translator = str.maketrans('', '', string.punctuation)
    return text.translate(translator)

with open("data/comb.txt", "r", encoding="utf-8") as f:
    input_data = f.read()
data = custom_standardization(input_data)

tokens = data.split()
print("Total number of tokens:", len(tokens))

In [ ]:
# Count word frequencies
word_counts = Counter(tokens)

# Assign indices starting from 1 (same as Keras)
vocab = {word: idx for idx, (word, _) in enumerate(word_counts.items(), start=1)}

# Reverse mapping
word2idx = {word: idx for idx, (word, _) in enumerate(vocab.items())}
idx2word = {idx: word for word, idx in word2idx.items()}
vocab_size = len(word2idx)
print("Vocab size is:", vocab_size)

sequence = torch.tensor(
    [word2idx[word] for word in tokens],
    dtype=torch.long
)

In [6]:
class CBOWDataset(Dataset):
    def __init__(self, sequence, window_size):
        self.data = []
        self.window_size = window_size

        for i in range(window_size, len(sequence) - window_size):
            context = torch.cat((
                sequence[i - WINDOW_SIZE:i],
                sequence[i + 1:i + WINDOW_SIZE + 1]
            ))
            target = sequence[i]
            self.data.append((context, target))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        context, target = self.data[idx]
        return context, target



class CBOWModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim
        )

        self.fc = nn.Linear(embedding_dim, vocab_size)

    def forward(self, x):
        """
        x shape: (batch_size, 2 * WINDOW_SIZE)
        """
        # Embedding lookup
        embeds = self.embedding(x)  
        # embeds shape: (batch_size, 2 * WINDOW_SIZE, embedding_dim)

        # Mean over context words (equivalent to tf.reduce_mean(axis=1))
        mean_embeds = embeds.mean(dim=1)
        # shape: (batch_size, embedding_dim)

        # Linear layer
        out = self.fc(mean_embeds)
        # shape: (batch_size, vocab_size)

        return out

In [7]:
dataset = CBOWDataset(sequence, WINDOW_SIZE)

In [12]:
model = CBOWModel(vocab_size, EMBEDDING_SIZE)
model = model.to(DEVICE)

In [ ]:
# # Uncomment it to load the pretrained model
# model.load_state_dict(torch.load(f"model/{MODEL_FILE}"))

In [ ]:
epochs = 10
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

for epoch in range(epochs):
    start = time.perf_counter()
    
    total_loss = 0
    for context, target in dataloader:
        context, target = context.to(DEVICE), target.to(DEVICE) # move to GPU
        
        optimizer.zero_grad()
        logits = model(context)
        loss = criterion(logits, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    end = time.perf_counter()

    print(f" ---- Epoch {epoch+1}, Loss: {total_loss:.4f}, Execution time: {end - start:.6f}")

In [15]:
# # Uncomment it to save the pretrained model
# torch.save(model.state_dict(), f"model/{MODEL_FILE}")

In [ ]:
word_embeddings = model.embedding.weight.detach()

# Example prediction
sentense = "Not to do what I".lower()
context_words = []

split_sentense = sentense.split(" ")

if (len(split_sentense) - 1) != WINDOW_SIZE * 2:
    print("ERROR: sentense does not have the сorrect lenght for testinng")
else:
    for w in split_sentense:
        context_words.append(w)
    del context_words[WINDOW_SIZE]

print("Context words: ", context_words)

In [ ]:
top_k = 5
model = model.to(DEVICE)
model.eval()

context_idxs = torch.tensor(
    [[word2idx[w] for w in context_words]]
)
context_idxs = context_idxs.to(DEVICE)

with torch.no_grad():
    logits = model(context_idxs)
    probs = torch.softmax(logits, dim=1)

top_idxs = torch.topk(probs, top_k).indices[0]
result = [idx2word[idx.item()] for idx in top_idxs]
print(result)

In [ ]:
v1 = word_embeddings[word2idx["he"]]
v2 = word_embeddings[word2idx["she"]]
F.cosine_similarity(v1, v2, dim=0).item()
# 0.5040047764778137

In [ ]:
def convert(arr):
    threshold = 0.01
    new_arr = []
    for i in arr:
        if abs(i) >= threshold:
            new_arr.append(0.0)
        else:
            new_arr.append(1.0)
            
    return torch.tensor(new_arr)

w1, w2 = "he", "she"
# w2 = "she"

idx1 = word2idx[w1]
idx2 = word2idx[w2]

word_emb1 = convert(word_embeddings[idx1])
word_emb2 = convert(word_embeddings[idx2])

F.cosine_similarity(word_emb1, word_emb2, dim=0).item()
# 0.40824827551841736